# Method 0 [Baseline]: TF-IDF + Soft Topics + Meta Features

這份 Notebook 將我們之前討論的所有測試流程（前處理、萃取 Meta 特徵、LDA 軟性主題萃取）整合在一起，並建立出我們的第一套基礎模型 (Baseline)！

In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.sparse import hstack

import warnings
warnings.filterwarnings('ignore')

# 載入資料 (因為目前在 EDA/ 資料夾下，所以要回上一層 ../ 找 datasets)
train_df = pd.read_csv('../datasets/train_2022.csv')
test_df = pd.read_csv('../datasets/test_2022.csv')
sample_sub = pd.read_csv('../datasets/sample_submission.csv')

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

Train Shape: (2000, 3)
Test Shape: (11000, 2)


In [17]:
# 1. 雙軌制前處理 A：針對 TF-IDF (移除無意義符號，保留特定文字)
def clean_for_tfidf(text):
    if not isinstance(text, str):
        return ""
    text = str(text).lower()
    
    # 清理括號與特殊標記
    text = text.replace('-lrb-', ' ').replace('-rrb-', ' ')
    
    # 只保留字母與必要標點符號，其餘過濾
    text = re.sub(r'[^a-zA-Z\s\?\!]', ' ', text)
    
    # 壓縮空白
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. Meta 特徵萃取 (捕捉標點符號的情緒暗示)
def extract_meta_features(df):
    meta_df = pd.DataFrame()

    meta_df['q_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('?'))
    meta_df['e_mark_count'] = df['TEXT'].apply(lambda x: str(x).count('!'))
    return meta_df

print("正在進行資料前處理...")
train_clean_text = train_df['TEXT'].apply(clean_for_tfidf)
test_clean_text = test_df['TEXT'].apply(clean_for_tfidf)

train_meta = extract_meta_features(train_df)
test_meta = extract_meta_features(test_df)
print("前處理完成！")

正在進行資料前處理...
前處理完成！


In [18]:
# 3. TF-IDF 轉換
print("建立 TF-IDF 矩陣...")
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
# 使用訓練集 fit，並對 train/test 進行 transform
X_train_tfidf = tfidf.fit_transform(train_clean_text)
X_test_tfidf = tfidf.transform(test_clean_text)

# 4. 利用 LDA 產生 Soft Topic 特徵 (設定 3 個 Topic 作為超參數)
print("透過 LDA 將 TF-IDF 轉換為 3 維 Soft Topic 特徵...")
n_topics = 3
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)

# 注意：LDA 吃的是文字頻率概念，雖然標準是 CountVector，但 TF-IDF 矩陣(非負)也能運作並產生機率分佈
X_train_topics = lda_model.fit_transform(X_train_tfidf)
X_test_topics = lda_model.transform(X_test_tfidf)

print(f"訓練集 Topic 機率矩陣大小: {X_train_topics.shape}")

建立 TF-IDF 矩陣...
透過 LDA 將 TF-IDF 轉換為 3 維 Soft Topic 特徵...
訓練集 Topic 機率矩陣大小: (2000, 3)


In [19]:
# 5. 組合所有特徵 (TF-IDF 矩陣 + Soft Topic 矩陣 + Meta 特徵矩陣)
print("組合最終特徵...")
X_train_final = hstack([X_train_tfidf, X_train_topics, train_meta.values])
X_test_final = hstack([X_test_tfidf, X_test_topics, test_meta.values])

y_train = train_df['LABEL'].values

print(f"最終特徵維度: {X_train_final.shape}")
print(f" (TF-IDF維度: {X_train_tfidf.shape[1]} + 軟主題維度: {X_train_topics.shape[1]} + Meta維度: {train_meta.shape[1]})")

組合最終特徵...
最終特徵維度: (2000, 2016)
 (TF-IDF維度: 2011 + 軟主題維度: 3 + Meta維度: 2)


In [21]:
# 6. 本地端交叉驗證 (Cross Validation) 預估分數
clf = LogisticRegression(max_iter=1000, random_state=42)
cv_scores = cross_val_score(clf, X_train_final, y_train, cv=5, scoring='accuracy')

print(f"5-Fold CV 準確率分數: {cv_scores}")
print(f"平均 CV 準確率: {cv_scores.mean():.4f}")

# 7. 訓練正式模型並產生 Kaggle 上傳檔案
clf.fit(X_train_final, y_train)
test_preds = clf.predict(X_test_final)

# 將預測結果與 test_df 的 row_id 結合，存回專案根目錄
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'LABEL': test_preds
})
submission.to_csv('../submission_baseline.csv', index=False)
print("模型訓練完畢，預測結果已存檔為 '../submission_baseline.csv'")

5-Fold CV 準確率分數: [0.635  0.63   0.6325 0.6625 0.6825]
平均 CV 準確率: 0.6485
模型訓練完畢，預測結果已存檔為 '../submission_baseline.csv'
